# FiLMVAE Training on Google Colab

Notebook simplificado para rodar o `src/filmvae/train.py` direto a partir de uma pasta no Google Drive.

Estrutura esperada dentro da pasta do projeto:
- `src/filmvae/train.py`
- `data/processed/train_filmvae_dataset_8.json`
- `data/processed/val_filmvae_dataset_8.json`
- `pyproject.toml`

Uso:
1. Copie a pasta do projeto para o Google Drive.
2. Ajuste apenas `PROJECT_DIR`, `RUN_MODE` e, se necessário, `WANDB_AGENT_COMMAND`.
3. Execute as células na ordem.

Se `RUN_MODE = "single_train"`, o notebook chama o `train.py` com os parâmetros definidos aqui.
Se `RUN_MODE = "wandb_agent"`, o notebook executa exatamente o comando que você colar em `WANDB_AGENT_COMMAND`.

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess

PROJECT_DIR = Path("/content/drive/MyDrive/AeroGen")
RUN_MODE = "single_train"  # opções: "single_train" ou "wandb_agent"
WANDB_AGENT_COMMAND = ""  # ex.: poetry run wandb agent --count 1 matsouto/FilmCSTVAE/abc12345
POETRY_VERSION = "1.8.3"

TRAIN_DATASET = "train_filmvae_dataset_8.json"
VAL_DATASET = "val_filmvae_dataset_8.json"
ENABLE_WANDB = RUN_MODE == "wandb_agent"

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Pasta do projeto não encontrada em {PROJECT_DIR}."
    )

print(f"Projeto ativo: {PROJECT_DIR}")
print(f"Diretório atual do notebook: {Path.cwd()}")

In [ ]:
!python -m pip install --quiet --upgrade pip
!python -m pip install --quiet poetry=={POETRY_VERSION}
!cd {PROJECT_DIR} && poetry config virtualenvs.create false
!cd {PROJECT_DIR} && poetry install --no-interaction

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

In [ ]:
processed_dir = PROJECT_DIR / "data" / "processed"
train_path = processed_dir / TRAIN_DATASET
val_path = processed_dir / VAL_DATASET
metadata_path = processed_dir / "filmvae_dataset_metadata_8.json"
train_script = PROJECT_DIR / "src" / "filmvae" / "train.py"

for required_path in [train_script, train_path, val_path, PROJECT_DIR / "pyproject.toml"]:
    if not required_path.exists():
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {required_path}")

print("Arquivos encontrados:")
print(f"  train script: {train_script}")
print(f"  train dataset: {train_path}")
print(f"  val dataset:   {val_path}")

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print("\nMetadata do dataset:")
    print(json.dumps(metadata, indent=2, ensure_ascii=False))

if ENABLE_WANDB:
    print("WandB será usado apenas no comando executado, sem importar no notebook.")

In [ ]:
TRAIN_CONFIG = {
    "model": "filmcstvae",
    "epochs": 150,
    "batch_size": 32,
    "latent_dim": 16,
    "film_depth": 2,
    "npv": 8,
    "learning_rate": 1e-3,
    "clipnorm": 1.0,
    "weight_decay": 0.0,
    "warmup_epochs": 100,
    "target_beta": 0.01,
    "beta_annealing": "cyclical",
    "condition_columns": ["Cl", "alpha"],
    "seed": 42,
    "airfoils_to_plot": 16,
    "checkpoint_epochs": 10,
    "verbose": 1,
    "sweep": False,
    "dev_mode": not ENABLE_WANDB,
}

if RUN_MODE == "single_train":
    command = [
        "poetry",
        "run",
        "python",
        str(train_script),
        "--model",
        TRAIN_CONFIG["model"],
        "--epochs",
        str(TRAIN_CONFIG["epochs"]),
        "--batch-size",
        str(TRAIN_CONFIG["batch_size"]),
        "--latent-dim",
        str(TRAIN_CONFIG["latent_dim"]),
        "--film-depth",
        str(TRAIN_CONFIG["film_depth"]),
        "--npv",
        str(TRAIN_CONFIG["npv"]),
        "--learning-rate",
        str(TRAIN_CONFIG["learning_rate"]),
        "--clipnorm",
        str(TRAIN_CONFIG["clipnorm"]),
        "--weight-decay",
        str(TRAIN_CONFIG["weight_decay"]),
        "--warmup-epochs",
        str(TRAIN_CONFIG["warmup_epochs"]),
        "--target-beta",
        str(TRAIN_CONFIG["target_beta"]),
        "--beta-annealing",
        TRAIN_CONFIG["beta_annealing"],
        "--seed",
        str(TRAIN_CONFIG["seed"]),
        "--airfoils-to-plot",
        str(TRAIN_CONFIG["airfoils_to_plot"]),
        "--checkpoint-epochs",
        str(TRAIN_CONFIG["checkpoint_epochs"]),
        "--verbose",
        str(TRAIN_CONFIG["verbose"]),
        "--condition-columns",
        *TRAIN_CONFIG["condition_columns"],
    ]

    if TRAIN_CONFIG["sweep"]:
        command.append("--sweep")

    if TRAIN_CONFIG["dev_mode"]:
        command.append("--dev")
else:
    if not WANDB_AGENT_COMMAND.strip():
        raise ValueError(
            "Preencha WANDB_AGENT_COMMAND quando RUN_MODE = 'wandb_agent'."
        )
    command = shlex.split(WANDB_AGENT_COMMAND)

print("Modo de execução:", RUN_MODE)
print("Comando:")
print(" ".join(shlex.quote(part) for part in command))

TRAIN_CONFIG

In [ ]:
process = subprocess.Popen(
    command,
    cwd=PROJECT_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_output = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
    captured_output.append(line)

return_code = process.wait()
print(f"\nExecução concluída com código: {return_code}")

if return_code != 0:
    raise RuntimeError("Comando falhou. Veja o log acima.")

if RUN_MODE == "single_train":
    model_root = PROJECT_DIR / "models" / TRAIN_CONFIG["model"]
    run_dirs = sorted(path for path in model_root.iterdir() if path.is_dir())
    if not run_dirs:
        raise FileNotFoundError(f"Nenhuma pasta de execução encontrada em {model_root}")

    latest_run = run_dirs[-1]
    print(f"Última execução: {latest_run}")

    for subdir in ["weights", "images", "scaler"]:
        subdir_path = latest_run / subdir
        if subdir_path.exists():
            files = sorted(path.name for path in subdir_path.iterdir())
            print(f"\n{subdir_path.name}:")
            for name in files[:10]:
                print(f"  - {name}")
            if len(files) > 10:
                print(f"  ... ({len(files) - 10} arquivos adicionais)")
else:
    print("Agent executado com streaming de logs no output da célula.")